# 🌳 ShadowRadix — 为混合注意力设计的前缀缓存

**本文目标**：深入理解 DeepSeek-V4 的 ShadowRadix——如何为混合注意力（SWA+C4/C128）实现原生前缀缓存。

读完这篇你会理解：
- 为什么普通的前缀缓存 (APC/RadixAttention) 在混合注意力下失效
- 虚拟 token 槽 (Virtual Token Slots) 和影子投影 (Shadow Projection) 的工作原理
- 双重计数器锁 (SWA lock / Full lock) 和 Tombstoning 机制
- ShadowRadix 与 SGLang RadixAttention 的继承和创新关系

## 1. 问题: 混合注意力如何做前缀缓存？

### 1.1 普通前缀缓存的假设被打破

```
vLLM APC / SGLang RadixAttention 的前提:
  每个 token 有一个统一的 KV Cache
  → 前缀相同 → KV Cache 相同 → 可以共享

DeepSeek-V4 的现实:
  每个 token 有三套 KV Cache!
  - SWA KV: 最近 128 个原始 token 的 K,V
  - C4 KV: 4:1 压缩后的全局 K,V
  - C128 KV: 128:1 压缩后的全局 K,V

  问题:
  1. 三个池 (pool) 的数据格式不同 (原始/4:1/128:1)
  2. 前缀 token 在不同池中的位置索引不同
  3. SWA 是滑动窗口 → 当前缀超过 128 tokens 时, 旧的 SWA 被淘汰
     但 C4/C128 的压缩 KV 还在!
```

### 1.2 具体例子

```
请求 A: "You are a helpful assistant. Please help me with..."
  SWA:   [最近的 128 tokens 的原始 KV]
  C4:    [每 4 token 压缩为 1 个] → top-512
  C128:  [每 128 token 压缩为 1 个] → 全量

请求 B: "You are a helpful assistant. What is the weather?"
  → 前缀 "You are a helpful assistant." 与 A 相同
  → 但 SWA / C4 / C128 中的表示方式不同!

如果 B 到达时 A 已经在 decode 了:
  A 的 SWA 窗口已经滑过 "You are..." → SWA KV 中不包含此前缀!
  但 A 的 C4/C128 压缩池中仍包含此前缀的压缩表示

问题: 如何让 B 复用 A 已计算的 KV Cache?
  → 不能简单复制 (SWA 已丢失)
  → 不能忽略 (C4/C128 仍然有效)
  → 需要一种机制来协调三个池
```

## 2. ShadowRadix 的核心设计

### 2.1 虚拟 Token 槽 — "统一的坐标系"

```
ShadowRadix 的核心思想:

创建一个 "虚拟全 token 槽" (virtual full-token slots) 的 radix tree
→ 作为所有 token 在整个序列中的"逻辑地址空间"
→ 就像 OS 虚拟内存给每个进程一个独立的地址空间

      Virtual Token Slots (Radix Tree)
      ┌───┬───┬───┬───┬───┬───┬───┬───┐
      │ 0 │ 1 │ 2 │ 3 │...│N-2│N-1│ N │  ← 逻辑 token 位置
      └─┬─┴─┬─┴─┬─┴─┬─┴───┴───┴───┴───┘
        │   │   │   │
        ▼   ▼   ▼   ▼
   ┌────────────────────────────────────┐
   │     Shadow Projections (影子投影)    │
   │                                    │
   │  SWA Pool:  [?, ?, tok2, tok3, ...] │ ← 只有最近的 128
   │  C4 Pool:   [c0, ?, c1, ?, ...]     │ ← 4:1 压缩
   │  C128 Pool: [C0, ?, ?, ...]          │ ← 128:1 压缩
   │                                    │
   │  ? = 该 token 在此池中不存在          │
   └────────────────────────────────────┘

每个虚拟 token 槽 "投影" 出它在三个物理池中的位置 (或 ∅)
→ 影子 (shadow) = 该 token 在某个池中的索引映射
```

### 2.2 影子投影的工作原理

```python
# 概念模型

class VirtualTokenSlot:
    """虚拟 token 槽 — Radix Tree 节点"""
    token_id: int
    logical_pos: int  # 在序列中的逻辑位置
    
    # 影子: 在每个物理池中的索引
    swa_shadow: Optional[int]    # SWA 池中的位置 (可能为 None)
    c4_shadow: Optional[int]     # C4 池中的位置
    c128_shadow: Optional[int]   # C128 池中的位置

class ShadowRadixCache:
    def __init__(self):
        self.radix_tree = RadixTree()  # 虚拟 token 槽的 radix tree
        self.swa_pool = PhysicalPool("SWA", capacity=128*N)
        self.c4_pool = PhysicalPool("C4", capacity=...)
        self.c128_pool = PhysicalPool("C128", capacity=...)
    
    def insert(self, tokens, kv_data):
        """插入一个新序列"""
        for i, token in enumerate(tokens):
            slot = self.radix_tree.insert(token, logical_pos=i)
            
            # 为每个池创建影子
            if self.is_in_swa_window(i, len(tokens)):
                slot.swa_shadow = self.swa_pool.allocate()
            if i % 4 == 0:  # C4 压缩
                slot.c4_shadow = self.c4_pool.allocate()
            if i % 128 == 0:  # C128 压缩
                slot.c128_shadow = self.c128_pool.allocate()
    
    def find_prefix(self, tokens):
        """查找前缀 → 返回已命中的虚拟 token 槽"""
        return self.radix_tree.match(tokens)
```

### 2.3 双重计数器锁

```
问题的另一面: 什么情况下可以释放 KV Cache?

SWA 池:
  - SWA 只在最近 128 tokens 内有效
  - 当 token 位置 > current_pos - 128 → SWA shadow 过期 → 释放

C4/C128 池:
  - 压缩 KV 对整个序列有效 (没有窗口限制)
  - 只要虚拟 token 槽还在被引用 → 不能释放

ShadowRadix 使用两个计数器:

  full_lock_ref: 保护源节点及其 C4/C128 影子
    → full_lock_ref > 0: 此节点的压缩 KV 不可释放
    
  swa_lock_ref: 仅跟踪节点是否在滑动窗口内
    → swa_lock_ref > 0: SWA 影子仍被引用
    → swa_lock_ref == 0: SWA 影子可以释放
```

### 2.4 Tombstoning (墓碑机制)

```
Tombstoning: 当 SWA 计数归零时保留节点的"骨架"

具体流程:
1. 某个 token 位置超出了所有活跃请求的 SWA 窗口
2. swa_lock_ref 降为 0
3. → SWA 槽位被释放 (标记为可用)
4. → 但虚拟 token 槽节点保留在 radix tree 中!
5. → C4/C128 影子继续有效
6. → 后续请求仍可通过压缩影子匹配前缀
7. → full_lock_ref 归零时 → 整个节点 (包括压缩影子) 被回收

这就是 "Tombstoning": 
  SWA 数据死了, 但墓碑还在 (压缩影子依然可以指引前缀匹配)
  
类比:
  图书馆里一本书 (SWA 原始数据) 被借走了
  但目录卡片 (C4/C128 压缩索引) 还在
  → 其他读者可以通过卡片知道这本书的存在和位置
  → 虽然不能直接看原书, 但可以通过摘要了解内容
```

## 3. ShadowRadix vs SGLang RadixAttention

| 维度 | SGLang RadixAttention | ShadowRadix |
|------|----------------------|-------------|
| 适用注意力 | Dense/Full Attention | **Hybrid Sparse (SWA+C4/C128)** |
| KV Cache 类型 | 单一 (每层一套 KV) | **三套 (SWA/C4/C128)** |
| 树节点 | 直接存 KV 数据 | **虚拟槽 + 影子指针** |
| 窗口管理 | 不需要 (Full Attention 无窗口) | **Tombstoning: SWA 过期但压缩保留** |
| 引用计数 | 单一 ref_count | **双锁: swa_lock_ref + full_lock_ref** |
| 前缀粒度 | Token-level | Token-level (继承自 RadixAttention) |
| 投机解码 | 不涉及 | **环大小翻倍** (防止回滚覆盖) |

### 3.1 投机解码的特殊处理

```
DeepSeek-V4 使用单层 MTP 头做投机解码 (speculative decoding)

问题:
  投机解码会"猜测"多个 future token
  如果猜错了 → 需要回滚 (discard 猜错的 token)
  
  如果回滚覆盖了之前缓存的 KV → 可能损坏 ShadowRadix!

ShadowRadix 的应对:
  → 投机解码时把环大小 (ring size) 翻倍
  → C4: 8 → 16
  → C128: 128 → 256
  → 更大的环 = 更多的缓冲空间
  → 回滚不会覆盖活跃数据
```

## 4. 总结: ShadowRadix 的三大创新

1. **虚拟 Token 槽作为统一坐标系**: 解决了多池之间的索引不一致问题
2. **双重锁 + Tombstoning**: 解决了 SWA 过期但压缩仍有效的问题
3. **投机解码环扩展**: 解决了回滚覆盖问题

这些设计使得 DeepSeek-V4 成为**第一个在混合注意力架构上实现原生前缀缓存的模型**。